In [ ]:
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

In [ ]:
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])


In [ ]:
from torchvision.datasets import ImageFolder

train_data = ImageFolder(root='dataset/train', transform=transform)
test_data = ImageFolder(root='dataset/test', transform=transform)


In [ ]:
train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
image, label = train_data[0]
image.size()

In [ ]:
class_names = train_data.classes
class_names

In [ ]:
class NeuralNet(nn.Module):
    def __init__(self, num_classes=5):
        super(NeuralNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*24*24, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


In [ ]:
net = NeuralNet()
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.002)

In [ ]:
for epoch in range(60):
    print(f'Training  epoch {epoch}...')
    
    running_loss = 0.0
    
    for i, data in enumerate(train_loader):
        inputs, labels = data
        
        optimizer.zero_grad()
        
        outputs = net(inputs)
        
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    print(f'Loss: {running_loss / len(train_loader):.4f}')

In [ ]:
torch.save(net.state_dict(), 'model2.pth')

In [ ]:
net = NeuralNet()
net.load_state_dict(torch.load('model2.pth'))

In [ ]:
correct = 0
total = 0

net.eval()

with torch.no_grad():
    for data in test_loader:
        images, labels = data
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
print(f'Accuracy: {100 * correct / total:.2f}%')

In [ ]:
new_tranform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

def load_image(image_path):
    image = Image.open(image_path)
    image = new_tranform(image).unsqueeze(0)
    return image

image_paths = ['dataset/validation/train/1.png', 'dataset/validation/motorcycle/4.png', 'dataset/validation/truck/3.png']

images = [load_image(image_path) for image_path in image_paths]

net.eval()
with torch.no_grad():
    for i, image in enumerate(images):
        outputs = net(image)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(probs, 1)
        print(f'Image {i+1} is predicted as {class_names[predicted.item()]}')
        for idx, class_name in enumerate(class_names):
            print(f'  {class_name}: {probs[0, idx].item()*100:.2f}%')
        print()
